<div class="alert alert-block alert-info">

# Part 5: Machine Learning: Random Forest Model

In this activity, you will build a predictive model using the same aromatase data. However, you will use a random forest classifier and MACCS Keys as descriptors.

Read the followng documents about random forest models to get a better understanding of this programming does
- https://scikit-learn.org/stable/modules/ensemble.html#forest
- https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier).

**Step 1**: Previously you saved the downsampled MACCS Keys descriptors and activity data for use later. We reload them to create your training and test sets.

In [ ]:
import numpy as np                                   # load numpy library

X_train = np.load("X_train_downsampled_MACCS167.npy")   # downsampled balanced set MACCS keys values
y_train = np.load("y_train_downsampled_MACCS167.npy")   # downsampled balanced set activity values
X_test = np.load("X_test_MACCS167.npy")                 # test set MACCS keys values
y_test = np.load("y_test_MACCS167.npy")                       # test set activity values

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)


**Step 2** Loading our previously saved best parameters

In `08_4_decision_tree_activity.ipynb`, we used cross-validation to identify the best decision tree parameters, including max_depth, min_samples_split, and min_samples_leaf. We saved those optimized parameters in a file called "best_DT_params.joblib" so they can be reloaded and reused in this notebook. Since a random forest is built from many individual decision trees, we can use those optimized decision tree settings as a starting point for the random forest model. This allows us to focus the new grid search on the number of trees in the forest, n_estimators, which makes the model-building process more efficient while still using parameters that were selected based on previous cross-validation results.

In [ ]:
# Load the best parameters from the Decision tree we previously generated

import joblib

best_params = joblib.load("best_DT_params.joblib")

print(best_params)

**Step 3** Building a Random Forest model using the balanced training data set.

You will be using: 
- Use 10-fold cross validation to select the best value for the "n_estimators" parameter that maximizes the **balanced accuracy**.  Test 40 values from 5 to 200 with an increment of 5 (e.g., 5, 10, 15, 20, ..., 190, 195, 200).
- For parameters 'max_depth', 'min_samples_leaf', and 'min_samples_split', use the best values found in `best_DT_params.joblib`.
- For other parameters, use the default values.
- For each parameter value, print the mean balanced accuracies (for both training and test from cross validation).

In [ ]:
# This cell will take a minute or two to run
# Random Forest with GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import numpy as np
import pandas as pd

# This handles either a regular GridSearchCV or a pipeline GridSearchCV
best_max_depth = best_params.get("max_depth", best_params.get("clf__max_depth"))
best_min_samples_leaf = best_params.get("min_samples_leaf", best_params.get("clf__min_samples_leaf"))
best_min_samples_split = best_params.get("min_samples_split", best_params.get("clf__min_samples_split"))

print("Using CV decision tree best parameters:")
print("max_depth:", best_max_depth)
print("min_samples_leaf:", best_min_samples_leaf)
print("min_samples_split:", best_min_samples_split)

# ------------------------------------------------------------
# Set up the Random Forest model inside a pipeline
# ------------------------------------------------------------

pipe_RF = Pipeline([
    ("var", VarianceThreshold(threshold=0.0)),
    ("clf", RandomForestClassifier(
        random_state=0,
        max_depth=best_max_depth,
        min_samples_leaf=best_min_samples_leaf,
        min_samples_split=best_min_samples_split
    ))
])

# Test 40 values from 5 to 200 in increments of 5
n_estimators_range = np.arange(5, 205, 5)

param_grid_RF = {
    "clf__n_estimators": n_estimators_range
}

# ------------------------------------------------------------
# Grid search using 10-fold cross-validation
# balanced_accuracy is used to select the best n_estimators value
# ------------------------------------------------------------

clf_RF_CV = GridSearchCV(
    estimator=pipe_RF,
    param_grid=param_grid_RF,
    cv=10,
    scoring="balanced_accuracy",
    refit=True,
    return_train_score=True
)

clf_RF_CV.fit(X_train, y_train)

print()
print("Best n_estimators:", clf_RF_CV.best_params_["clf__n_estimators"])
print("Best cross-validation balanced accuracy:", clf_RF_CV.best_score_)

# ------------------------------------------------------------
# Print the mean balanced accuracy for each n_estimators value
# ------------------------------------------------------------

results_RF = pd.DataFrame(clf_RF_CV.cv_results_)

summary_RF = results_RF[[
    "param_clf__n_estimators",
    "mean_train_score",
    "mean_test_score"
]].copy()

summary_RF = summary_RF.rename(columns={
    "param_clf__n_estimators": "n_estimators",
    "mean_train_score": "Mean Training Balanced Accuracy",
    "mean_test_score": "Mean CV Test Balanced Accuracy"
})

print()
print(summary_RF.to_string(index=False))


# Save the randome forest pipeline
joblib.dump(clf_RF_CV, "ds_maccs_RF_pipeline.joblib")

**Step 4** Apply the developed RF model to predict the activity of the **training** set compounds.

- Report the confusion matrix.
- Report the accuracy, balanced accuracy, sensitivity, specificity, and auc-roc.

In [ ]:
from sklearn.metrics import classification_report #provides detailed report that includes precision and sensitivity
from sklearn.metrics import confusion_matrix      # gives a 2x2 matrix for true netatives, false positives, false negatives and true positives
from sklearn.metrics import accuracy_score        # computes accuracy = number of correct preictions/total number of preditions
from sklearn.metrics import roc_auc_score         # Computs Area under the ROC curve, evaluates trade-off of true positive rate and false positive rate

#predict the activities of the training set
y_true, y_pred = y_train, clf_RF_CV.predict(X_train)

# generate confusion matrix
CMat = confusion_matrix( y_true, y_pred )   
print(CMat)    # [[TN, FP], 
               #  [FN, TP]]
# Extracting TN, FP, FN, TP from the confusion matrix               
TN = CMat[0, 0]  # True Negatives
FP = CMat[0, 1]  # False Positives
FN = CMat[1, 0]  # False Negatives
TP = CMat[1, 1]  # True Positives

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print("Total predictions:", TN + FP + FN + TP)
print()
print()

acc  = accuracy_score( y_true, y_pred ) #correct predictions/total predictions = (TP+TN)/(TP+TN+FP+FN)
prec = TP / (TP + FP)    # precision = TP / (TP + FP) measures how well model identifies actual positives
sens = TP / (FN + TP)    # sensitivity = TP / (FN + TP) measures how well model identifies actual positives
spec = TN / (TN + FP)    # specificity = TN / (TN + FP)measures how well model identifies actual negatives
bacc = (sens + spec) / 2    #averages sensititivy and specificity
f1_score = 2 * (prec * sens) / (prec + sens)  # F1 score is the harmonic mean of precision and sensitivity

y_score = clf_RF_CV.predict_proba( X_train )[:, 1] #returns proability of each class, [:, 1] extracts the probability of the positive class (label 1) for each sample.
auc = roc_auc_score( y_true, y_score ) #measures how well the model ranks psitive vs negative samples. AUC = 1.0 → perfect model; AUC = 0.5 → random guessing.

print("Training set performance metrics:")
print(f"Accuracy          = {acc:.4f}") # how accurate is the model overall
print(f"Precision         = {prec:.4f}") # How well it identifies actual positives (precision)
print(f"Sensitivity       = {sens:.4f}") # How well it catches positives (sensitivity)
print(f"Specificity       = {spec:.4f}") # How well it avoids false negatives (specificity)
print(f"Balanced Accuracy = {bacc:.4f}") # How balanced its performance is across classes
print(f"F1 Score          = {f1_score:.4f}") # Harmonic mean of precision and sensitivity
print(f"AUC-ROC           = {auc:.4f}")  #How well it ranks predictions (AUC)

**Step 5** Apply the developed RF model to predict the activity of the **test** set compounds.

- Report the accuracy, balanced accurayc, sensitivity, specificity, and auc-roc.

In [ ]:
#predict the activities of the training set
y_true, y_pred = y_test, clf_RF_CV.predict(X_test)

# generate confusion matrix
CMat = confusion_matrix( y_true, y_pred )   
print(CMat)    # [[TN, FP], 
               #  [FN, TP]]
# Extracting TN, FP, FN, TP from the confusion matrix               
TN = CMat[0, 0]  # True Negatives
FP = CMat[0, 1]  # False Positives
FN = CMat[1, 0]  # False Negatives
TP = CMat[1, 1]  # True Positives

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print("Total predictions:", TN + FP + FN + TP)
print()
print()

acc  = accuracy_score( y_true, y_pred ) #correct predictions/total predictions = (TP+TN)/(TP+TN+FP+FN)
prec = TP / (TP + FP)    # precision = TP / (TP + FP) measures how well model identifies actual positives
sens = TP / (FN + TP)    # sensitivity = TP / (FN + TP) measures how well model identifies actual positives
spec = TN / (TN + FP)    # specificity = TN / (TN + FP)measures how well model identifies actual negatives
bacc = (sens + spec) / 2    #averages sensititivy and specificity
f1_score = 2 * (prec * sens) / (prec + sens)  # F1 score is the harmonic mean of precision and sensitivity

y_score = clf_RF_CV.predict_proba( X_test )[:, 1] #returns proability of each class, [:, 1] extracts the probability of the positive class (label 1) for each sample.
auc = roc_auc_score( y_true, y_score ) #measures how well the model ranks psitive vs negative samples. AUC = 1.0 → perfect model; AUC = 0.5 → random guessing.

print("Training set performance metrics:")
print(f"Accuracy          = {acc:.4f}") # how accurate is the model overall
print(f"Precision         = {prec:.4f}") # How well it identifies actual positives (precision)
print(f"Sensitivity       = {sens:.4f}") # How well it catches positives (sensitivity)
print(f"Specificity       = {spec:.4f}") # How well it avoids false negatives (specificity)
print(f"Balanced Accuracy = {bacc:.4f}") # How balanced its performance is across classes
print(f"F1 Score          = {f1_score:.4f}") # Harmonic mean of precision and sensitivity
print(f"AUC-ROC           = {auc:.4f}")  #How well it ranks predictions (AUC)



**Step 6** Test this model on your previous selected molecules

In [ ]:
# load necessary libraries
from rdkit import Chem
from rdkit.DataStructs import ConvertToNumpyArray
from rdkit.Chem import MACCSkeys


#using new pipeline to predict
pipe = joblib.load("ds_maccs_RF_pipeline.joblib")
print("Pipeline loaded successfully!")
print()

In [ ]:
#Define new SMILES string
new_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O" # CID 2242 aspirin should be INactive
#new_smiles = "C1=CC=C(C=C1)C(C2=CC=CC=C2)(C3=CC=CC=C3Cl)N4C=CN=C4" # CID = 2812 should be active
#new_smiles = "C1=CC=C(C(=C1)C2=NC(=NO2)C3=CC=NC=C3)Cl" #CID 65758 should be active
#new_smiles = "C1=CC(=CC=C1C2=COC3=CC(=CC(=C3C2=O)O)O)O" #CID 5280961 should be active
#new_smiles = "CN(C1CCN(CC1)C2=NC3=CC=CC=C3N2CC4=CC=C(C=C4)F)C5=NC=CC(=O)N5" #CID 65906 should be INactive
#new_smiles = "C1=CNC(=O)NC1=O" #CID 1174 should be INactive
#new_smiles = "CCCCCC1=CC(=C2C=CC(OC2=C1)(C)CCC=C(C)C)O" #CID30219 not in datbase (CBC)
#new_smiles = "C[C@H]1C[C@@H](C(=O)[C@@H](C1)[C@@H](CC2CC(=O)NC(=O)C2)O)C" #CID 6197 should be active
#new_smiles = "CCCCCC1=CC(=C2[C@@H]3C=C(CC[C@H]3C(OC2=C1)(C)C)C)O" #CID16078 in database and should be active (THC)
#new_smiles = "COC1=CC(=CC(=C1OC)OC)CCN" #CID4076 not in database (mescaline)
#new_smiles = "CC1=C(C(CCC1)(C)C)/C=C/C(=C/C=C/C(=C/CO)/C)/C" # vitamin A not in database
mol = Chem.MolFromSmiles(new_smiles)
mol

In [ ]:

fp = MACCSkeys.GenMACCSKeys(mol)

# prepare the array based on the fingerprint
X_query = np.array(fp).reshape(1, -1) # The classifier needs a 2D shape, but we have 1D list. We reshape (1, n_bits)
pred = pipe.predict(X_query)  # no manual masking needed
if pred == 0:
    print("Molecule is inactive for human aromatase.")
else:
    print("Molecule is active for human aromatase. Active may be agonist or antagonist in this model.")

# Returns the model's confidence for each possible class (e.g., [P(Inactive), P(Active)]).
probability = pipe.predict_proba(X_query)

print("Probabilities (Inactive, Active):", probability)

In [ ]:
# We can also loop through a list to predict multiple activities.

#new_smiles = "C1=CC=C(C(=C1)C2=NC(=NO2)C3=CC=NC=C3)Cl" #CID 65758 should be active
#new_smiles = "C1=CC(=CC=C1C2=COC3=CC(=CC(=C3C2=O)O)O)O" #CID 5280961 should be active
#new_smiles = "CN(C1CCN(CC1)C2=NC3=CC=CC=C3N2CC4=CC=C(C=C4)F)C5=NC=CC(=O)N5" #CID 65906 should be INactive
#new_smiles = "C1=CNC(=O)NC1=O" #CID 1174 should be INactive
#new_smiles = "C[C@H]1C[C@@H](C(=O)[C@@H](C1)[C@@H](CC2CC(=O)NC(=O)C2)O)C" #CID 6197 should be active

smiles_list = ["C1=CC=C(C(=C1)C2=NC(=NO2)C3=CC=NC=C3)Cl", "C1=CC(=CC=C1C2=COC3=CC(=CC(=C3C2=O)O)O)O", 
               "CN(C1CCN(CC1)C2=NC3=CC=CC=C3N2CC4=CC=C(C=C4)F)C5=NC=CC(=O)N5","C1=CNC(=O)NC1=O", 
               "C[C@H]1C[C@@H](C(=O)[C@@H](C1)[C@@H](CC2CC(=O)NC(=O)C2)O)C"]
fps = []
for s in smiles_list:
    mol = Chem.MolFromSmiles(s)
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=int)
    ConvertToNumpyArray(fp, arr)
    fps.append(arr)

X_batch = np.array(fps)
preds = pipe.predict(X_batch)
print(preds)
preds = pipe.predict_proba(X_batch)
print(preds)